### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


/tmp/ipykernel_23112/3805701412.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
def process_all_pdf(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    #print(pdf_files)

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata 
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")
    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents

#process all pdfs in the data directory
all_pdf_documents = process_all_pdf("../data")



Found 3 PDF files to process

Processing: Cal Poly Slo Decision Letter.pdf
 Loaded 1 pages

Processing: Alice B. Hansen Scholarship Form.pdf
 Loaded 1 pages

Processing: written assignment 3.pdf
 Loaded 5 pages

 Total documents loaded: 7


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Technolutions', 'creator': 'Slate', 'creationdate': '2025-06-08T00:53:36-04:00', 'author': '', 'title': 'Cal Poly', 'moddate': '2025-06-08T00:53:36-04:00', 'source': '../data/pdf/Cal Poly Slo Decision Letter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Cal Poly Slo Decision Letter.pdf', 'file_type': 'pdf'}, page_content="1 GRAND AVENUE • SAN LUIS OBISPO • CALIFORNIA • 934 07                                       CALPOLY.EDU  \n \n \nApril 1, 2025 \n \nVictor Xie \n1516 Buena Vista Ave Apt A \nAlameda, CA 94501-1218\nCongratulations Victor,\nOur answer is, yes! You did it! You applied, we accepted and so it is my privilege to offer you\nconditional admission to the fall 2025 quarter at Cal Poly, where Learn by Doing has inspired and\nguided students toward lifelong success since 1901. (And where 95% of Mustangs are employed or in\ngraduate school within 9 months of graduation.)\nIn the Computer Science major, you will apply your 

In [ ]:
###Text splitting (get into chunks)

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better Rag Performances"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ["\n\n", "\n", " ",""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        printf